# Missing Data and Imputing

**Notebook 3 of 6.** You have a dataset that no longer lies about its types, its units or
its categories. It still has holes in it. This notebook is about finding them, working out
whether they fall in a pattern, and deciding what to put in their place.

By the end you should be able to:

- count missing values per column and visualise them with `missingno`.
- judge whether missing values occur at random or in a pattern, and why that matters.
- drop or impute them with a constant, a summary statistic, or interpolation.
- say what each choice costs.

> **Note.** `06_mutual_information.ipynb` measures what these choices did to the conclusions you end
> up drawing. The decisions you make here are not neutral.

First the libraries this notebook needs.

In [ ]:
# data analysis
import numpy as np
import pandas as pd

# missing value visualisation
import missingno as msno

Then the data, as the previous notebook left it.

In [ ]:
# the cleaned data from 01_data_cleaning
df_weather = pd.read_csv("../data/seattle-weather_clean.csv", parse_dates=["date"])
df_weather.head()

Missing data is a common problem when working with real data sets. The reasons for missing data can be manifold, e.g. sensor failures, old data, improper data management and even human error. 

It is important that missing data is identified and dealt with appropriately before further data analysis or machine learning. Many machine learning algorithms cannot handle missing data and require that entire rows or columns where a single missing value is present be deleted. Or missing data has to be replaced (imputed) with a new value.

Knowing when missing data occurs can help avoid missing values in the future, or help in dealing with them.
There is a great python library for doing some EDA specifically for missing data, called **[missingno](https://github.com/ResidentMario/missingno)**. (Do not confuse it with the pokemon of the same name.)

In [ ]:
# import missingno
import missingno as msno

We can have a look at how many values are missing per column in our data.

In [ ]:
# display number of missing values per column
df_weather.isna().sum()

With missingno we can simply display these values as missing percentages per column in a bar chart. Each bar represents one column. On the left side we see the percentage of data present, on the right side we see the exact number of values that correspond to these percentages.

In [ ]:
# plotting percentage of missing values per column
msno.bar(df_weather)

We see that only two columns contain missing values: More than 20 % of the data is missing in the column "temp_max". Less than 10 % of the data is missing in the column "precipitation". It would be interesting to know if there is a pattern in the occurrence of the missing values. This would help us understand where they are coming from. For example, it could be that these values are not collected on Sunday. Then we would have a missing value in every 7th row. It could also show us a period when our instrument stopped working, which would then lead to successive missing values. All this helps us to understand our data better and to correct errors if necessary.

With missingno you can also print a matrix plot. For each row in the dataset you can see if data is missing in each column. When data is present, the plot is shaded in grey (or your color of choice), and when it is absent the plot is displayed in white. This gives you an view how that missing data is distributed in the data frame. And if they occur together.

In [ ]:
msno.matrix(df_weather)

On the right hand side you can see for each data point how many columns contain data. In our case, the range is from 4 columns with data (left) to 6 columns with data (right). As you can see, there are not many rows where missing values occur together. And overall, there is no clear pattern in the missing data. Therefore, we have to assume, that the missing data occurs randomly. Which is not helping us at all on how to best treat this missing values. 

Missingno has more plots to investigate missing values. For example, you can also create a heatmap to see how often missing values in specific features occur together. Check out the repo for more insights!

### Imputing Data

Missing Data and Imputing are big topics in data science. Imputing means to replace missing values with sensible values. There are several imputing techniques that we will familiarize ourselves with in this notebook. When you impute missing values, you are manipulating the data and this can have a significant impact on the quality of the data.  Hence you need to have a certain level of knowledge about your data in order to find a good solution and the approach that is best suited for the task at hand. As usual in data science, there is no one solution that fits all problems. Imputation of data ranges from fairly simple to highly complex strategies. Remember that the work of a data scientist progresses in cycles. It is always best to start with a simple approach and become more complex with each cycle. With this in mind, your pre-processing will only become more complex and thus more time-consuming if it really provides more value.

The first step is always to determine how many missing data you got. Since we already identified all our missing data above, we only have to use one command.

> **Exercise**: Time for brainstorming! What could be reasons for missing values and what different ways of dealing with missing data can you think of? Are these possibilities dependent on the reasons of missing values?

**Explanation**

Missing values arise for many reasons: sensor or instrument failure, data not collected on certain days, fields left blank on a form, merge or export errors, or values that simply do not apply to a record. Statisticians group these into three mechanisms: MCAR (missing completely at random), MAR (missing at random, explainable by other observed columns), and MNAR (missing not at random, where the missingness depends on the unseen value itself). The ways to deal with them include dropping rows or columns, imputing with a constant or with the mean, median or mode, forward or backward filling, interpolating, and predicting the value with a model. The right choice does depend on the mechanism: dropping or simple mean imputation can be fine when data is MCAR and the share is small, but for MAR or MNAR a method that uses the related columns is safer, since blunt imputation would bias the data.

### Dropping data

Dropping data is the easiest and fastest option. But this is only recommended if there’s a lot of data to start with and the percentage of missing values is low. There are multiple ways to drop your data. You could drop the rows which have missing values in them or even drop a whole column.

Though you could end up with little data if you have too much missing data. You also could loose some (valuable) information or maybe evan change the representation of some data if the rows with missing values have something in common and are not totally random. Hence you should always be mindful about dropping your data.

In our case we have only 2 columns with missing values and the amount doesn't seem too much. But let's put those numbers in context.

In [ ]:
print(f"numbers of rows : {df_weather.shape[0]}")
print(
    f"missing values in precipitation : {round(df_weather.precipitation.isna().sum() / df_weather.shape[0] * 100, 2)} %"
)
print(
    f"missing values in temp_max : {round(df_weather.temp_max.isna().sum() / df_weather.shape[0] * 100, 2)} %"
)
print(
    f"missing values in data frame : {round(df_weather.isna().sum().sum() / (df_weather.shape[0] * df_weather.shape[1]) * 100, 2)} %"
)

We have 1461 rows and about one quarter of them have missing values in at least one column. This would actually be too much to drop. Fortunately there are more ways of imputing. But for a start dropping is still a possibility.

Pandas has a function for dropping missing values. You will find a description and an explanation for a few attributes in the following table:

`dropna` | description
---|---
`.dropna()` | drops row if contains at least 1 NaN
`.dropna(how='all')` | drops row if all entries are filled with NaN
`.dropna(thresh=2)` | drops row if there are less than 2 cells filled with a valid value

You can also check out the [documentation for `dropna`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.dropna.html).

> **Exercise:** Since we also want to show you other ways of imputing in this notebook, make a copy of our dataset. After that use `.dropna` on our copied data set. Try at least two different ways.

In [ ]:
# create a copy of the dataset so the cleaned df_weather stays untouched
df_dropna = df_weather.copy()

# way 1: drop a row if it contains at least one missing value (in any column)
df_dropna_any = df_dropna.dropna()

# way 2: drop a row only if every value in it is missing
df_dropna_all = df_dropna.dropna(how="all")

print(f"rows before dropping        : {df_weather.shape[0]}")
print(f"rows after dropna()         : {df_dropna_any.shape[0]}")
print(f"rows after dropna(how='all'): {df_dropna_all.shape[0]}")

**Explanation**

We work on `.copy()` so the cleaned `df_weather` is never mutated by the experiment. `.dropna()` removes any row that has at least one NaN, which here drops every row missing precipitation or temp_max. `.dropna(how="all")` only removes rows where every column is NaN, so it changes nothing in this dataset, and printing the row counts side by side makes that difference obvious. With about a quarter of the rows affected, plain `.dropna()` would throw away too much data, which is exactly why the imputing methods below are worth trying.

### Imputing with a constant

#### Flag the missing value (optional)

Sometimes even a missing value can be informative. For example if you asking a proband for their sex and give them only the option of ticking female or male and their define them selfs as queer, it might conclude in a missing value. You could assume and assign that value to `other` or another categorical value.
For our case we could assign a unique value. For precipitation and temperature this would be a really high absolute value.

#### Mean/Median/Mode imputation

A third option is to fill each gap with a summary of the column it sits in: the **mean**,
the **median** or the **mode**. Which of the three depends on the column. A mean is
meaningless for a categorical variable, and on a numerical one it is dragged around by
extreme values, so the median is often the safer choice. Sometimes an assumption about the
data beats all three, for example that today's weather resembles yesterday's.

Pandas covers these with `fillna` and the fill methods:

`method` | description
---|---
`df.fillna(0)` | replaces every NaN in df with one value
`df.fillna({'precipitation': 0, 'temp_max': 42})` | replaces NaN in named columns with specific values
`df.ffill()` | use the value of the row before (forward fill)
`df.bfill()` | use the value of the row after (backward fill)
`df.ffill(limit=1)` | fill only the first gap in a consecutive run, leave the rest NaN

See the documentation for
[`fillna`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.fillna.html)
and [`bfill`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.bfill.html).

Two of them applied to this data, for comparison with the dropping you just did:

In [ ]:
# method 1: fill the numeric columns with their median (robust to outliers, unlike the mean)
df_fillna = df_weather.copy()
df_fillna["precipitation"] = df_fillna["precipitation"].fillna(
    df_fillna["precipitation"].median()
)
df_fillna["temp_max"] = df_fillna["temp_max"].fillna(df_fillna["temp_max"].median())

# method 2: forward fill, carrying the previous day's value into the gap
df_ffill = df_weather.copy().ffill()

print("missing values after median fill:")
print(df_fillna.isna().sum())
print("\nmissing values after forward fill:")
print(df_ffill.isna().sum())

**Why these two.** Median imputation fills each numeric column with its own median, chosen
over the mean because precipitation and temperature contain extreme values that would drag
a mean upwards. Forward fill instead carries the previous day's value into the gap, which
is a reasonable assumption for daily weather because consecutive days resemble each other.

Note `.ffill()` rather than the older `fillna(method="ffill")`, which is deprecated. The
`weather` column is categorical, so a median means nothing there: forward fill, the mode,
or an explicit constant are the options for it.

> **Note.** `06_mutual_information.ipynb`, the optional fifth notebook, comes back to these choices
> and measures what each of them did to the conclusions you draw from the data.

#### Interpolation

The final method in this notebook is calculating the missing values with an interpolation. An interpolation estimates the course of a continuous function between two known points. From this function we can approximate the missing value.

There are multiple ways of interpolation. The most common and also the easiest for a missing value $x_{i}$ is the linear interpolation, where two data points, $(x_{i-1},y_{i-1})$ and $(x_{i+1},y_{i+1})$, are connected by a straight line.

$$
y_{i} = y_{i-1} + \frac{y_{i+1} - y_{i-1}}{x_{i+1} - x_{i-1}}(x_i - x_{i-1})
$$

In our case the maximum temperature is a perfect example. If we use the default settings of pandas `.interpolate` we would get a linear interpolation with an equal spaced distribution. It is also possible to take the date instead.

> **Exercise:** Please make a copy from `df_weather` again and try to do an interpolation for the missing values in the maximum temperature column. You might want to check out the [documentation for `interpolate`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.interpolate.html).

In [ ]:
# create a copy of the dataset
df_interpolate = df_weather.copy()

# linear interpolation estimates each missing value from the points on either side of the gap
df_interpolate["temp_max"] = df_interpolate["temp_max"].interpolate()

print(f"missing temp_max before: {df_weather['temp_max'].isna().sum()}")
print(f"missing temp_max after : {df_interpolate['temp_max'].isna().sum()}")

**Explanation**

`.interpolate()` defaults to linear interpolation, drawing a straight line between the known values on either side of each gap and reading the missing value off that line. This suits temp_max because temperature changes gradually and the rows are in date order, so neighbouring days are good predictors. It cannot fill a gap at the very start of the column, since there is no earlier point to interpolate from, so a leading NaN can remain. For unevenly spaced gaps you could pass `method="time"` after setting the date as the index, which weights the estimate by how far apart the surrounding dates are.

Since it is also possible to use splines and polynomials, the interpolation function can be quite powerful. Reading further documentation is absolute necessary to use this function!

### Prediction (optional)

It is also possible to predict your missing values. Therefore the column with the missing data would become the target value. The dataset will be split into a set with a valid value for the target which would be used for training and a set with the missing values, for which we would predict. A good model for imputing is KNN. Since we didn't reach the subject of predictive modeling yet, this will only be mentioned at this point.

> **Note.** It is important to prevent any data leakage! When using a model to impute missing data make sure you remove your actual target values! If you impute temp_max, you cannot use the weather columns when your end goal is to predict the weather.

## Wrapping up

You now have four ways of dealing with a gap, and no rule that picks between them for you:

| Approach | Reach for it when | What it costs |
|---|---|---|
| **drop** the rows | little is missing and you have data to spare | rows, and whatever those rows had in common |
| a **constant** or a flag | the absence is itself informative | a value that is not real, which everything downstream treats as if it were |
| **mean, median or mode** | the column has a stable centre | spread, because every filled value sits exactly at the centre |
| **interpolation** | rows are ordered and neighbours resemble each other | nothing obvious, which is exactly why it is easy to over-trust |

Two things to carry forward. **Missing data is rarely missing at random**, so look for the
pattern with `missingno` before choosing. And **there is no neutral option**: doing nothing
is a choice too, and dropping a quarter of your rows has consequences of its own.

> **This notebook deliberately saves nothing.** Every technique above was applied to a
> copy, so `seattle-weather_clean.csv` still has its gaps and the next notebooks start from
> the same place you did. Choosing an imputation is a decision about your analysis, not a
> tidying step, and the rest of the day shows you what it costs before you commit.

Next: `04_descriptive_statistics.ipynb` stops repairing the data and starts asking what it
says.